<a href="https://colab.research.google.com/github/npd00/dnd_2024_character_sheet_generator/blob/main/DND_2024_Character_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D&D 2024 Character Generator (AI-Powered)
**Version 1.0 | Compatible with D&D 2024 Player's Handbook (5.5e)**

This notebook generates complete level 1 characters using:
*   **Official 2024 Rules**: Species, Backgrounds, Origin Feats, Weapon Mastery.
*   **Google Gemini API**: For rich, unique character backstories and descriptions.
*   **WeasyPrint**: To generate professional PDF character sheets.

## 🔑 Setup: Gemini API Key

This notebook uses Google's Gemini API (completely free, no credit card required).

### Step 1: Get Your Free API Key
1. Go to: https://aistudio.google.com/app/apikey
2. Sign in with your Google account
3. Click "Create API Key"
4. Copy the API key

### Step 2: Add API Key to Colab Secrets
1. Click the 🔑 key icon in the left sidebar (Secrets)
2. Click "+ Add new secret"
3. Name: `GEMINI_API_KEY`
4. Value: Paste your API key
5. Toggle "Notebook access" ON for this notebook

**Important:** Never paste your API key directly in code cells. Always use Colab Secrets.


In [1]:
!pip install -U -q google-generativeai weasyprint jinja2
!apt-get install -y -q libpango-1.0-0 libpangoft2-1.0-0
print("✓ Dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 14.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
libpango-1.0-0 is already the newest version (1.50.6+ds-2ubuntu1).
libpango-1.0-0 set to manually installed.
libpangoft2-1.0-0 is already the newest version (1.50.6+ds-2ubuntu1).
libpangoft2-1.0-0 set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✓ Dependencies installed


In [2]:
import google.generativeai as genai
from google.colab import userdata
import random
import json
from jinja2 import Template
from weasyprint import HTML

# Configure API
try:
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("✓ Gemini API configured successfully!")
except Exception as e:
    print("⚠️ WARNING: GEMINI_API_KEY not found in Colab Secrets.")
    print("AI Narrative generation will be disabled.")
    api_key = None


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ Gemini API configured successfully!


## Core D&D 2024 Rules Engine

In [3]:
import random
import json
from typing import List, Dict, Optional, Union

# =============================================================================
# D&D 2024 RULES: DATA CONSTANTS
# =============================================================================

# SPECIES DATA
# Notes:
# - ASIs are now tied to Backgrounds, not Species.
# - All species have base speed 30ft (unless noted, but 2024 standardized this for core species).
SPECIES_DATA = {
    "Human": {
        "speed": 30,
        "size": "Medium or Small",
        "darkvision": 0,
        "traits": [
            "Resourceful: You gain Heroic Inspiration whenever you finish a Long Rest.",
            "Skillful: You gain proficiency in one skill of your choice.",
            "Versatile: You gain one Origin Feat of your choice."
        ]
    },
    "Elf": {
        "speed": 30,
        "size": "Medium",
        "darkvision": 60,
        "traits": [
            "Fey Ancestry: Advantage on saves to end Charmed condition.",
            "Keen Senses: Proficiency in Insight, Perception, or Survival.",
            "Trance: You don't sleep; meditate for 4 hours for Long Rest.",
            "Elven Lineage: You are part of a magical lineage."
        ]
    },
    "Dwarf": {
        "speed": 30,
        "size": "Medium",
        "darkvision": 120,
        "traits": [
            "Dwarven Resilience: Resistance to Poison damage; Advantage on saves vs Poisoned.",
            "Dwarven Toughness: HP Max increases by 1 per level.",
            "Stonecunning: Tremorsense 60ft on stone (Bonus Action) for 10 min."
        ]
    },
    "Halfling": {
        "speed": 30,
        "size": "Small",
        "darkvision": 0,
        "traits": [
            "Brave: Advantage on saves vs Frightened.",
            "Halfling Nimbleness: Move through space of creatures larger than you.",
            "Luck: Reroll natural 1s on d20 tests (must use new roll).",
            "Naturally Stealthy: You can Hide when obscured by a creature larger than you."
        ]
    },
    "Dragonborn": {
        "speed": 30,
        "size": "Medium",
        "darkvision": 60,
        "traits": [
            "Breath Weapon: Exhale magical energy (Line or Cone). Dex save for damage.",
            "Damage Resistance: Resistant to damage type associated with ancestry.",
            "Draconic Flight: (Level 5) Temporary flight capability."
        ]
    },
    "Orc": {
        "speed": 30,
        "size": "Medium",
        "darkvision": 120,
        "traits": [
            "Adrenaline Rush: Dash as Bonus Action.",
            "Relentless Endurance: Drop to 1 HP instead of 0 (once per Long Rest)."
        ]
    },
    "Tiefling": {
        "speed": 30,
        "size": "Medium or Small",
        "darkvision": 60,
        "traits": [
            "Otherworldly Presence: Know Thaumaturgy cantrip.",
            "Fiendish Legacy: Gain resistance and spells based on legacy (Abyssal, Chthonic, or Infernal)."
        ]
    }
}

# BACKGROUND DATA
# 2024 Rules:
# - Grants +2 to one ability, +1 to another (from list of 3).
# - Grants 1 Origin Feat.
# - Grants 2 Skill Proficiencies.
# - Grants 1 Tool Proficiency.
# - Grants 50 GP (standardized for this generator).
BACKGROUND_DATA = {
    "Acolyte": {
        "abilities": ["Intelligence", "Wisdom", "Charisma"],
        "feat": "Magic Initiate (Cleric)",
        "skills": ["Insight", "Religion"],
        "tool": "Calligrapher's Supplies"
    },
    "Criminal": {
        "abilities": ["Dexterity", "Constitution", "Intelligence"],
        "feat": "Alert",
        "skills": ["Sleight of Hand", "Stealth"],
        "tool": "Thieves' Tools"
    },
    "Guard": {
        "abilities": ["Strength", "Intelligence", "Wisdom"],
        "feat": "Alert",
        "skills": ["Athletics", "Perception"],
        "tool": "Gaming Set"
    },
    "Noble": {
        "abilities": ["Strength", "Intelligence", "Charisma"],
        "feat": "Skilled",
        "skills": ["History", "Persuasion"],
        "tool": "Gaming Set"
    },
    "Sage": {
        "abilities": ["Constitution", "Intelligence", "Wisdom"],
        "feat": "Magic Initiate (Wizard)",
        "skills": ["Arcana", "History"],
        "tool": "Calligrapher's Supplies"
    },
    "Sailor": {
        "abilities": ["Strength", "Dexterity", "Wisdom"],
        "feat": "Tavern Brawler",
        "skills": ["Acrobatics", "Perception"],
        "tool": "Navigator's Tools"
    },
    "Soldier": {
        "abilities": ["Strength", "Dexterity", "Constitution"],
        "feat": "Savage Attacker",
        "skills": ["Athletics", "Intimidation"],
        "tool": "Gaming Set"
    },
    "Wayfarer": { # Replaces Urchin
        "abilities": ["Dexterity", "Wisdom", "Charisma"],
        "feat": "Lucky",
        "skills": ["Insight", "Stealth"],
        "tool": "Thieves' Tools"
    }
}

# CLASS DATA
# 2024 Rules:
# - Weapon Mastery for martial classes.
# - Level 1 Subclasses for some (Warlock/Sorcerer/Cleric - Cleric gets Divine Order).
CLASS_DATA = {
    "Fighter": {
        "hit_die": 10,
        "primary_ability": ["Strength", "Dexterity"],
        "saves": ["Strength", "Constitution"],
        "armor": ["All Armor", "Shields"],
        "weapons": ["Simple Weapons", "Martial Weapons"],
        "tools": [],
        "features": [
            "Fighting Style: Choose a specialized style of combat.",
            "Second Wind: Regain Hit Points (Bonus Action).",
            "Weapon Mastery: Mastery properties for 3 weapons."
        ],
        "spellcasting": False,
        "weapon_mastery": True
    },
    "Wizard": {
        "hit_die": 6,
        "primary_ability": ["Intelligence"],
        "saves": ["Intelligence", "Wisdom"],
        "armor": [],
        "weapons": ["Simple Weapons"],
        "tools": [],
        "features": [
            "Spellcasting: Cast Wizard spells (Int based).",
            "Ritual Adept: Cast ritual spells from book.",
            "Arcane Recovery: Regain spell slots on Short Rest."
        ],
        "spellcasting": True,
        "weapon_mastery": False
    },
    "Rogue": {
        "hit_die": 8,
        "primary_ability": ["Dexterity"],
        "saves": ["Dexterity", "Intelligence"],
        "armor": ["Light Armor"],
        "weapons": ["Simple Weapons", "Martial Weapons (Finesse properties)"],
        "tools": ["Thieves' Tools"],
        "features": [
            "Sneak Attack: Extra damage on advantage/flanking.",
            "Thieves' Cant: Secret language of rogues.",
            "Weapon Mastery: Mastery properties for 2 weapons.",
            "Expertise: Double proficiency in two skills."
        ],
        "spellcasting": False,
        "weapon_mastery": True
    },
    "Cleric": {
        "hit_die": 8,
        "primary_ability": ["Wisdom"],
        "saves": ["Wisdom", "Charisma"],
        "armor": ["Light Armor", "Medium Armor", "Shields"],
        "weapons": ["Simple Weapons"],
        "tools": [],
        "features": [
            "Spellcasting: Cast Cleric spells (Wis based).",
            "Divine Order: Choose Protector (Armor) or Thaumaturge (Extra Cantrip)."
        ],
        "spellcasting": True,
        "weapon_mastery": False
    },
    "Ranger": {
        "hit_die": 10,
        "primary_ability": ["Dexterity", "Wisdom"],
        "saves": ["Strength", "Dexterity"],
        "armor": ["Light Armor", "Medium Armor", "Shields"],
        "weapons": ["Simple Weapons", "Martial Weapons"],
        "tools": [],
        "features": [
            "Spellcasting: Cast Ranger spells (Wis based).",
            "Favored Enemy: Learn Hunter's Mark (Free casts).",
            "Weapon Mastery: Mastery properties for 2 weapons.",
            "Deft Explorer: Expertise in one skill."
        ],
        "spellcasting": True,
        "weapon_mastery": True
    },
    "Barbarian": {
        "hit_die": 12,
        "primary_ability": ["Strength"],
        "saves": ["Strength", "Constitution"],
        "armor": ["Light Armor", "Medium Armor", "Shields"],
        "weapons": ["Simple Weapons", "Martial Weapons"],
        "tools": [],
        "features": [
            "Rage: Resistance to physical damage, damage bonus.",
            "Unarmored Defense: AC = 10 + Dex + Con.",
            "Weapon Mastery: Mastery properties for 2 weapons."
        ],
        "spellcasting": False,
        "weapon_mastery": True
    },
    "Bard": {
        "hit_die": 8,
        "primary_ability": ["Charisma"],
        "saves": ["Dexterity", "Charisma"],
        "armor": ["Light Armor"],
        "weapons": ["Simple Weapons"],
        "tools": ["3 Musical Instruments"],
        "features": [
            "Spellcasting: Cast Bard spells (Cha based).",
            "Bardic Inspiration: Grant bonus dice to allies.",
        ],
        "spellcasting": True,
        "weapon_mastery": False
    }
}

ORIGIN_FEATS_DATA = {
    "Alert": "Initiative Bonus (+PB). Can swap initiative with ally.",
    "Magic Initiate (Cleric)": "Learn 2 Cleric cantrips and 1 Level 1 Cleric spell.",
    "Magic Initiate (Wizard)": "Learn 2 Wizard cantrips and 1 Level 1 Wizard spell.",
    "Skilled": "Gain proficiency in 3 Skills or Tools.",
    "Tavern Brawler": "Unarmed Strike d4 damage. Push 5ft on hit. Improvised weapon proficiency.",
    "Savage Attacker": "Reroll weapon damage dice (use higher).",
    "Lucky": "Luck Points to gain advantage or impose disadvantage.",
    "Crafter": "Gain tool proficiencies and crafting discount.", # Not in base backgrounds list but good to have
    "Healer": "Reroll healing dice. Stabilize to 1 HP.",
    "Musician": "Grant Inspiration when finishing Short Rest.",
    "Tough": "+2 HP per level."
}

# =============================================================================
# LOGIC
# =============================================================================

class Character:
    def __init__(self, name="Adventurer", species="Human", class_name="Fighter", background="Soldier", player_name="Player"):
        self.name = name
        self.player_name = player_name
        self.species = species
        self.class_name = class_name
        self.background = background
        self.level = 1

        # Stats
        self.ability_scores = {"Strength": 10, "Dexterity": 10, "Constitution": 10,
                               "Intelligence": 10, "Wisdom": 10, "Charisma": 10}
        self.modifiers = {}
        self.proficiencies = {
            "skills": [],
            "saves": [],
            "armor": [],
            "weapons": [],
            "tools": [],
            "languages": ["Common"]
        }
        self.derived_stats = {
            "hp": 0,
            "ac": 10,
            "initiative": 0,
            "speed": 30,
            "proficiency_bonus": 2
        }
        self.features = []
        self.equipment = []
        self.spells = {
            "cantrips": [],
            "level_1": []
        }
        self.narrative = {
            "physical": "",
            "personality": [],
            "ideal": "",
            "bond": "",
            "flaw": "",
            "backstory": ""
        }

    def generate_ability_scores(self, method="standard_array"):
        """Generates ability scores, applies background bonuses, and updates modifiers."""
        # 1. Base Scores
        scores = {}
        primary = CLASS_DATA[self.class_name]["primary_ability"]

        if method == "standard_array":
            # Priority assignment: Primary -> Con/Dex -> Others
            standard = [15, 14, 13, 12, 10, 8]

            # Assign highest to primary
            p_ability = primary[0]
            scores[p_ability] = standard.pop(0)

            # Smart assignment for the rest
            priority_order = ["Constitution", "Dexterity", "Wisdom", "Strength", "Charisma", "Intelligence"]

            # Fill remaining abilities
            for ability in priority_order:
                if ability not in scores:
                    scores[ability] = standard.pop(0)

        elif method == "roll":
            # 4d6 drop lowest
            rolls = []
            for _ in range(6):
                dice = [random.randint(1, 6) for _ in range(4)]
                dice.sort()
                rolls.append(sum(dice[1:]))
            rolls.sort(reverse=True)

            # Same priority assignment logic
            p_ability = primary[0]
            scores[p_ability] = rolls.pop(0)
            priority_order = ["Constitution", "Dexterity", "Wisdom", "Strength", "Charisma", "Intelligence"]
            for ability in priority_order:
                if ability not in scores:
                    scores[ability] = rolls.pop(0)

        # 2. Apply Background Bonuses (2024 Rule)
        # Choose +2/+1 from background options
        bg_options = BACKGROUND_DATA[self.background]["abilities"]
        # Logic: +2 to Primary if available, else best fit
        bonus_2 = None
        bonus_1 = None

        # Try to put +2 in primary
        for ability in primary:
            if ability in bg_options:
                bonus_2 = ability
                break

        # If primary not in background options, pick the first option as +2
        if not bonus_2:
            bonus_2 = bg_options[0]

        # Pick +1 (must be different)
        remaining_opts = [x for x in bg_options if x != bonus_2]
        # Try to put +1 in Con or Dex if possible
        for ability in ["Constitution", "Dexterity"]:
            if ability in remaining_opts:
                bonus_1 = ability
                break
        if not bonus_1:
            bonus_1 = remaining_opts[0]

        scores[bonus_2] += 2
        scores[bonus_1] += 1

        # Cap at 20 (though 18 is soft cap at lvl 1 usually)

        self.ability_scores = scores

        # 3. Calculate Modifiers
        for ability, score in scores.items():
            self.modifiers[ability] = (score - 10) // 2

    def calculate_derived_stats(self):
        """Calculates HP, AC, Initiative based on stats and class."""
        c_data = CLASS_DATA[self.class_name]

        # HP
        con_mod = self.modifiers["Constitution"]
        hp = c_data["hit_die"] + con_mod
        if self.species == "Dwarf":
            hp += 1
        if "Tough" in self.features: # If they got the feat somehow
            hp += 2
        self.derived_stats["hp"] = hp

        # AC (Basic unarmored/light estimation)
        dex_mod = self.modifiers["Dexterity"]
        base_ac = 10 + dex_mod

        # Barbarian Unarmored Defense
        if self.class_name == "Barbarian":
            # 10 + Dex + Con
            base_ac = 10 + dex_mod + self.modifiers["Constitution"]

        # Heavy Armor users (Fighter/Paladin/Cleric-Protector) - simplified
        if "All Armor" in c_data["armor"]:
             base_ac = 16 # Chain Mail approximation
        elif "Medium Armor" in c_data["armor"]:
             base_ac = min(10 + dex_mod, 12) + 2 # Scale Mail (14) + Dex(max 2) - simplified
             if dex_mod >= 2: base_ac = 14
             else: base_ac = 12 + dex_mod

        self.derived_stats["ac"] = base_ac

        # Initiative
        init = dex_mod
        if self.background in ["Criminal", "Guard"] or "Alert" in self.features:
            # Alert feat gives +PB to initiative
            init += self.derived_stats["proficiency_bonus"]
        self.derived_stats["initiative"] = init

        # Speed
        self.derived_stats["speed"] = SPECIES_DATA[self.species]["speed"]

    def apply_features_and_proficiencies(self):
        """Aggregates all features from Species, Class, Background."""
        s_data = SPECIES_DATA[self.species]
        c_data = CLASS_DATA[self.class_name]
        b_data = BACKGROUND_DATA[self.background]

        # Proficiencies
        self.proficiencies["saves"] = c_data["saves"]
        self.proficiencies["armor"] = c_data["armor"]
        self.proficiencies["weapons"] = c_data["weapons"]

        # Skills: Class + Background + Human(optional)
        # For simplicity, we just add Background skills.
        # Real app would let user pick class skills. We will auto-pick class skills to avoid conflicts.
        self.proficiencies["skills"].extend(b_data["skills"])
        if self.species == "Human":
            # Add a random skill not already in list
            pass

        # Tools
        if b_data.get("tool"):
            self.proficiencies["tools"].append(b_data["tool"])
        self.proficiencies["tools"].extend(c_data["tools"])

        # Features
        self.features.extend(s_data["traits"])
        self.features.extend(c_data["features"])

        # Origin Feat
        feat_name = b_data["feat"]
        feat_desc = ORIGIN_FEATS_DATA.get(feat_name, "Feat description unavailable.")
        self.features.append(f"Origin Feat: {feat_name} - {feat_desc}")

        # Human Versatile Feat
        if self.species == "Human":
            self.features.append("Versatile: Extra Origin Feat (e.g., Skilled)")

    def full_generation(self):
        self.generate_ability_scores()
        self.apply_features_and_proficiencies()
        self.calculate_derived_stats()

# Test run
if __name__ == "__main__":
    hero = Character(name="Valeros", species="Human", class_name="Fighter", background="Soldier")
    hero.full_generation()
    print(f"Name: {hero.name}")
    print(f"Stats: {hero.ability_scores}")
    print(f"HP: {hero.derived_stats['hp']} AC: {hero.derived_stats['ac']}")
    print(f"Features: {hero.features}")


Name: Valeros
Stats: {'Strength': 17, 'Constitution': 15, 'Dexterity': 13, 'Wisdom': 12, 'Charisma': 10, 'Intelligence': 8}
HP: 12 AC: 16
Features: ['Resourceful: You gain Heroic Inspiration whenever you finish a Long Rest.', 'Skillful: You gain proficiency in one skill of your choice.', 'Versatile: You gain one Origin Feat of your choice.', 'Fighting Style: Choose a specialized style of combat.', 'Second Wind: Regain Hit Points (Bonus Action).', 'Weapon Mastery: Mastery properties for 3 weapons.', 'Origin Feat: Savage Attacker - Reroll weapon damage dice (use higher).', 'Versatile: Extra Origin Feat (e.g., Skilled)']


## PDF Generation Engine

In [4]:


import datetime

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <style>
        @page { size: A4; margin: 0.5in; }
        body { font-family: 'Helvetica', 'Arial', sans-serif; color: #333; line-height: 1.4; font-size: 11pt; }
        h1, h2, h3 { color: #8B0000; border-bottom: 2px solid #8B0000; margin-top: 20px; margin-bottom: 10px; }

        /* Header Table */
        .header-table { width: 100%; border-bottom: 3px solid #333; margin-bottom: 20px; border-collapse: collapse; }
        .header-table td { vertical-align: bottom; padding-bottom: 10px; }
        .char-name { font-size: 2.5em; font-weight: bold; font-family: 'Georgia', serif; color: #222; margin: 0; line-height: 1; }
        .header-label { font-size: 0.7em; text-transform: uppercase; color: #666; display: block; margin-top: 2px; }
        .header-val { font-size: 1.1em; font-weight: bold; display: block; border-bottom: 1px solid #ccc; min-width: 100px; }
        .header-info-group { text-align: left; padding: 0 10px; }

        .stats-grid { display: grid; grid-template-columns: repeat(6, 1fr); gap: 10px; text-align: center; margin-bottom: 20px; }
        .stat-box { border: 1px solid #333; padding: 5px; border-radius: 5px; background: #fff; }
        .stat-label { display: block; font-weight: bold; font-size: 0.8em; text-transform: uppercase; margin-bottom: 5px; }
        .stat-score { font-size: 1.6em; font-weight: bold; display: block; line-height: 1; }
        .stat-mod { font-size: 1em; background: #eee; border-radius: 10px; padding: 2px 8px; margin-top: 5px; display: inline-block; font-weight: bold; border: 1px solid #ccc; }

        /* Combat Stats Bar */
        .combat-table { width: 100%; border-collapse: separate; border-spacing: 10px 0; margin-bottom: 20px; }
        .combat-box { border: 2px solid #555; padding: 10px; text-align: center; border-radius: 8px; background: #f9f9f9; }
        .combat-val { font-size: 20px; font-weight: bold; display: block; }
        .combat-label { font-size: 10px; text-transform: uppercase; color: #555; display: block; margin-top: 5px; }

        .section-box { margin-bottom: 15px; }
        .narrative-text { font-style: italic; background: #fffdf5; padding: 15px; border: 1px solid #e0e0e0; border-radius: 5px; font-family: 'Georgia', serif; }

        table.list-table { width: 100%; border-collapse: collapse; margin-bottom: 10px; font-size: 0.9em; }
        table.list-table th, table.list-table td { border-bottom: 1px solid #ddd; padding: 6px; text-align: left; }
        table.list-table th { background-color: #f2f2f2; font-weight: bold; }

        ul { margin: 5px 0; padding-left: 20px; }
        li { margin-bottom: 3px; }
    </style>
</head>
<body>

    <table class="header-table">
        <tr>
            <td style="width: 40%;">
                <div class="char-name">{{ name }}</div>
                <div class="header-label">Character Name</div>
            </td>
            <td style="width: 60%;">
                <table style="width: 100%;">
                    <tr>
                        <td class="header-info-group">
                            <span class="header-val">{{ class_name }} 1</span>
                            <span class="header-label">Class & Level</span>
                        </td>
                        <td class="header-info-group">
                            <span class="header-val">{{ background }}</span>
                            <span class="header-label">Background</span>
                        </td>
                        <td class="header-info-group">
                            <span class="header-val">{{ player_name }}</span>
                            <span class="header-label">Player Name</span>
                        </td>
                    </tr>
                    <tr>
                        <td class="header-info-group">
                            <span class="header-val">{{ species }}</span>
                            <span class="header-label">Species</span>
                        </td>
                         <td class="header-info-group" colspan="2">
                             <!-- Space for alignment or extra fields -->
                        </td>
                    </tr>
                </table>
            </td>
        </tr>
    </table>

    <!-- Ability Scores -->
    <div class="stats-grid">
        {% for ability, score in abilities.items() %}
        <div class="stat-box">
            <span class="stat-label">{{ ability[:3].upper() }}</span>
            <span class="stat-score">{{ score }}</span>
            <span class="stat-mod">{{ "+" if modifiers[ability] >= 0 else "" }}{{ modifiers[ability] }}</span>
        </div>
        {% endfor %}
    </div>

    <!-- Combat Stats -->
    <table class="combat-table">
        <tr>
            <td class="combat-box">
                <span class="combat-val">{{ ac }}</span>
                <span class="combat-label">Armor Class</span>
            </td>
            <td class="combat-box">
                <span class="combat-val">{{ max_hp }}</span>
                <span class="combat-label">Hit Points</span>
            </td>
            <td class="combat-box">
                <span class="combat-val">{{ "+" if initiative >= 0 else "" }}{{ initiative }}</span>
                <span class="combat-label">Initiative</span>
            </td>
            <td class="combat-box">
                <span class="combat-val">{{ speed }} ft</span>
                <span class="combat-label">Speed</span>
            </td>
            <td class="combat-box">
                <span class="combat-val">+{{ prob_bonus }}</span>
                <span class="combat-label">Proficiency</span>
            </td>
        </tr>
    </table>

    <div style="display: flex; gap: 20px;">
        <!-- Left Column -->
        <div style="flex: 1;">
            <div class="section-box">
                <h3>Proficiencies</h3>
                <p><strong>Saving Throws:</strong> {{ saves|join(', ') }}</p>
                <p><strong>Skills:</strong> {{ skills|join(', ') }}</p>
                <p><strong>Tools:</strong> {{ tools|join(', ') }}</p>
                <p><strong>Languages:</strong> {{ languages|join(', ') }}</p>
                <p><strong>Armor:</strong> {{ armor|join(', ') }}</p>
                <p><strong>Weapons:</strong> {{ weapons|join(', ') }}</p>
            </div>

            <div class="section-box">
                <h3>Features & Traits</h3>
                <ul>
                {% for feature in features %}
                    <li>{{ feature }}</li>
                {% endfor %}
                </ul>
            </div>

            {% if spellcasting %}
            <div class="section-box">
                <h3>Spellcasting</h3>
                <p><strong>Primary Ability:</strong> {{ spell_ability }}</p>
                <p><strong>Save DC:</strong> {{ spell_save_dc }}</p>
                <p><strong>Attack Bonus:</strong> +{{ spell_attack }}</p>
            </div>
            {% endif %}
        </div>

        <!-- Right Column -->
        <div style="flex: 1;">
            <div class="section-box">
                <h3>Equipment</h3>
                <ul>
                {% for item in equipment %}
                    <li>{{ item }}</li>
                {% endfor %}
                    <li>50 GP (Background Starting Gold)</li>
                </ul>
            </div>

            <div class="section-box">
                <h3>Attacks</h3>
                <table class="list-table">
                    <thead><tr><th>Name</th><th>Bonus</th><th>Damage</th></tr></thead>
                    <tbody>
                        <tr><td>Unarmed Strike</td><td>+{{ modifiers['Strength'] + prob_bonus }}</td><td>1 + {{ modifiers['Strength'] }}</td></tr>
                        <!-- Placeholder for weapons -->
                        <tr><td>Primary Weapon</td><td>+{{ modifiers['Strength'] + prob_bonus }}</td><td>1d8 + {{ modifiers['Strength'] }}</td></tr>
                    </tbody>
                </table>
            </div>
        </div>
    </div>

    <div class="narrative-section">
        <h2>Character Narrative</h2>

        <div class="section-box">
            <h3>Appearance</h3>
            <p class="narrative-text">{{ narrative.physical }}</p>
        </div>

        <div class="section-box">
            <h3>Personality & Backstory</h3>
            <p><strong>Personality Traits:</strong> {{ narrative.personality|join(', ') }}</p>
            <p><strong>Ideal:</strong> {{ narrative.ideal }}</p>
            <p><strong>Bond:</strong> {{ narrative.bond }}</p>
            <p><strong>Flaw:</strong> {{ narrative.flaw }}</p>

            <h4>Backstory</h4>
            <div class="narrative-text" style="white-space: pre-wrap;">{{ narrative.backstory }}</div>
        </div>
    </div>

    <div style="text-align: center; margin-top: 30px; border-top: 1px solid #eee; padding-top: 10px; color: #888; font-size: 0.8em;">
        Generated by D&D 2024 Character Generator (Gemini AI) | 2024 Rules Compatible
    </div>

</body>
</html>
"""

def render_html(c: Character):
    """Renders the HTML template with character data."""
    # Jinja2 logic is simple enough to replace with simple string replacement or use jinja2 library
    # For robust implementation in Colab, we should use Jinja2.
    from jinja2 import Template

    template = Template(HTML_TEMPLATE)

    # Calculate some derived values for display
    spell_ability = c.class_name == "Wizard" and "Intelligence" or \
                    c.class_name in ["Cleric", "Ranger"] and "Wisdom" or \
                    c.class_name == "Bard" and "Charisma" or "None"

    spell_save_dc = 0
    spell_attack = 0
    if spell_ability != "None":
        mod = c.modifiers[spell_ability]
        spell_save_dc = 8 + c.derived_stats["proficiency_bonus"] + mod
        spell_attack = c.derived_stats["proficiency_bonus"] + mod

    return template.render(
        name=c.name,
        player_name=c.player_name,
        species=c.species,
        class_name=c.class_name,
        background=c.background,
        abilities=c.ability_scores,
        modifiers=c.modifiers,
        ac=c.derived_stats["ac"],
        max_hp=c.derived_stats["hp"],
        initiative=c.derived_stats["initiative"],
        speed=c.derived_stats["speed"],
        prob_bonus=c.derived_stats["proficiency_bonus"],
        saves=c.proficiencies["saves"],
        skills=c.proficiencies["skills"],
        tools=c.proficiencies["tools"],
        languages=c.proficiencies["languages"],
        armor=c.proficiencies["armor"],
        weapons=c.proficiencies["weapons"],
        features=c.features,
        equipment=c.equipment,
        narrative=c.narrative,
        spellcasting=c.class_name in ["Wizard", "Cleric", "Ranger", "Bard"],
        spell_ability=spell_ability,
        spell_save_dc=spell_save_dc,
        spell_attack=spell_attack
    )

def create_pdf(character, filename="character_sheet.pdf"):
    try:
        from weasyprint import HTML
        html_content = render_html(character)
        HTML(string=html_content).write_pdf(filename)
        print(f"PDF successfully created: {filename}")
        return True
    except ImportError:
        print("WeasyPrint not found. Please install it: pip install weasyprint")
        return False
    except Exception as e:
        print(f"Error creating PDF: {e}")
        return False


## AI Narrative Engine

In [5]:

import json
import os


# Placeholder for Colab Userdata
try:
    from google.colab import userdata
    HAS_COLAB = True
except ImportError:
    HAS_COLAB = False

def get_api_key():
    if HAS_COLAB:
        try:
            return userdata.get('GEMINI_API_KEY')
        except:
            return None
    else:
        # Fallback for local testing
        return os.environ.get('GEMINI_API_KEY')

def generate_narrative(character: Character):
    """Generates narrative using Gemini API."""
    import google.generativeai as genai

    api_key = get_api_key()
    if not api_key:
        print("Skipping AI generation: No API Key found.")
        # Fill with placeholders
        character.narrative = {
            "physical": "A sturdy adventurer ready for the road.",
            "personality": ["Determined", "Brave"],
            "ideal": "Justice",
            "bond": "I protect those who cannot protect themselves.",
            "flaw": "I never back down from a fight.",
            "backstory": "Born in a small village..."
        }
        return

    genai.configure(api_key=api_key)

    prompt = f"""
    You are a D&D 2024 expert creative writer. precise rules knowledge.
    Create a rich narrative description for this level 1 character:

    Name: {character.name}
    Species: {character.species}
    Class: {character.class_name}
    Background: {character.background}
    Stats: {character.ability_scores}
    Feats: {character.features}

    Please provide the output in valid JSON format with the following keys:
    - physical: (string) 3-4 sentences description.
    - personality: (list of 2 strings) traits.
    - ideal: (string) 1 sentence.
    - bond: (string) 1 sentence.
    - flaw: (string) 1 sentence.
    - backstory: (string) 3 paragraphs (Origins, Training, Motivation).

    Keep it strictly within the JSON structure.
    """

    # Updated list based on 2026 available models
    models_to_try = [
        'gemini-2.5-flash',
        'gemini-2.0-flash',
        'gemini-flash-latest',
        'gemini-2.5-pro',
        'gemini-2.0-flash-lite'
    ]

    for model_name in models_to_try:
        try:
            print(f"Attempting to generate with model: {model_name}...")
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
            data = json.loads(response.text)
            character.narrative = data
            print(f"✓ AI Narrative generated successfully using {model_name}.")
            return
        except Exception as e:
            print(f"Failed with {model_name}: {e}")
            last_error = e

    # If all fail
    print(f"Error generating narrative: {last_error}")
    character.narrative["physical"] = f"Error generating description: {str(last_error)}"
    character.narrative["backstory"] = "Please check your GEMINI_API_KEY in Colab Secrets and ensure you have access to Gemini models."


## Create Your Character

In [6]:
# @title Character Options
# @markdown Select your character options below:

player_name = "Player 1" # @param {type:"string"}
character_name = "Valeros" # @param {type:"string"}
species = "Human" # @param ["Human", "Elf", "Dwarf", "Halfling", "Dragonborn", "Orc", "Tiefling"]
class_name = "Fighter" # @param ["Fighter", "Wizard", "Rogue", "Cleric", "Ranger", "Barbarian", "Bard"]
background = "Soldier" # @param ["Acolyte", "Criminal", "Guard", "Noble", "Sage", "Sailor", "Soldier", "Wayfarer"]

# Create Character
hero = Character(name=character_name, species=species, class_name=class_name, background=background, player_name=player_name)
hero.full_generation()

print(f"✅ Character Created: {hero.name} (Played by {hero.player_name})")
print(f"   {hero.species} {hero.class_name} ({hero.background})")
print(f"   Stats: {hero.ability_scores}")
print(f"   HP: {hero.derived_stats['hp']} | AC: {hero.derived_stats['ac']}")


✅ Character Created: Valeros (Played by Player 1)
   Human Fighter (Soldier)
   Stats: {'Strength': 17, 'Constitution': 15, 'Dexterity': 13, 'Wisdom': 12, 'Charisma': 10, 'Intelligence': 8}
   HP: 12 | AC: 16


In [7]:
print("🔮 Consulting the Oracle (Gemini)...")
generate_narrative(hero)
print("✓ Narrative generated!")
print(f"Physical: {hero.narrative['physical']}")


🔮 Consulting the Oracle (Gemini)...
Attempting to generate with model: gemini-2.5-flash...
✓ AI Narrative generated successfully using gemini-2.5-flash.
✓ Narrative generated!
Physical: Valeros is a man forged by the rigors of military life, his frame broad and muscular, speaking volumes of his formidable strength and endurance. Numerous scars crisscross his forearms and knuckles, silent testaments to battles fought and won, or at least survived. His eyes, though often narrowed with a soldier's perpetual vigilance, hold a directness that brooks no nonsense, framed by a neatly trimmed, practical haircut. His posture remains military-straight, even in repose, a testament to years of discipline.


In [8]:
filename = f"{hero.name.replace(' ', '_')}_Sheet.pdf"
print(f"📄 Generating PDF: {filename}...")
create_pdf(hero, filename)

from google.colab import files
files.download(filename)


📄 Generating PDF: Valeros_Sheet.pdf...


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.004s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.008s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'glyf' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'glyf' tabl

PDF successfully created: Valeros_Sheet.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>